In [67]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import Markdown
from dotenv.ipython import load_dotenv

In [68]:
load_dotenv(override=True)

True

In [69]:
llm=ChatOpenAI(model="gpt-4o",temperature=0)

In [70]:
agent = create_agent(
model=llm,
system_prompt= "You are a helpful assistant"
)

In [71]:
agent = create_agent(
model="openai:gpt-4o",
tools=[],
system_prompt= "You are a helpful assistant"
)

In [73]:
respt = agent.invoke(input={"messages":
[{'role':'user', 'content':'je m appelle douaa'}]})

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [74]:
print(display(Markdown(resp['messages'][-1].content)))

Il semble que votre message soit incomplet. Pourriez-vous le compléter ou préciser votre demande afin que je puisse vous aider au mieux ?

None


In [75]:
from langchain.agents.middleware import wrap_model_call,ModelRequest, ModelResponse
from langchain.messages import HumanMessage,SystemMessage, AIMessage


In [76]:
basic_llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)
advanced_llm=ChatOpenAI(model="gpt-4o",temperature=0)

In [80]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.state.get("env", "test")
    print("ENV:", env)

    if env == "prod":
        print("advanced model selected")
        request.model = advanced_llm
    else:
        print("basic model selected")
        request.model = basic_llm

    return handler(request)

In [81]:
agent2 = create_agent(
model=basic_llm, 
tools=[],
middleware=[dynamic_model_selection],
debug=True
)

In [84]:
resp = agent2.invoke(
    {
        "messages": [HumanMessage(content="c'est quoi un agent ai?")],
        "env": "prod"  
    }
)
print(resp["messages"][-1].content)

[values] {'messages': [HumanMessage(content="c'est quoi un agent ai?", additional_kwargs={}, response_metadata={}, id='1655278b-5dec-458d-86a7-ecc7f3326b8a')]}
ENV: test
basic model selected


C:\Users\douaa\AppData\Local\Temp\ipykernel_14648\3921408860.py:11: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = basic_llm


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [85]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

In [88]:
agent = create_agent(
model="openai:gpt-4o",
system_prompt="You are a helpful assistant",
checkpointer=InMemorySaver()
)

In [87]:
res=agent.invoke(
input={'messages':[HumanMessage('My Name is Mohamed')]},
config={'configurable':{'thread_id':1}}
)
print(res['messages'][-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [89]:
res=agent.invoke(
input={'messages':[HumanMessage('What is my name')]},
config={'configurable':{'thread_id':1}}
)
print(res['messages'][-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [107]:
from langchain.tools import tool
from langchain.agents import create_agent

In [108]:
@tool
def get_weather(city: str) -> str:
  """Get weather information for a city."""
  print(f'Weather tool is called for {city}')
  return {
    'city':city,
    'temperature': 32,
    'humidity': 60,
    'condition': 'Sunny'
  }
@tool
def get_employee_info(name: str) -> str:
  """Get information aboud the given employee name"""
  print(f'get_employee_info tool is called for {name}')
  return {'name' : name, 
          'salary': 12000, 
          'job': 'Developper'}

In [ ]:
agent4 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant",
)

In [111]:
response=agent4.invoke(
    input={'messages':[HumanMessage('What is the weather in Marrakech')]},
    config={'configurable': {'thread_id': '1'}}
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
resp=agent4.invoke(input={'messages':[HumanMessage('What is the weather in Marrakech?')]})

In [ ]:
resp=agent4.invoke(
    input={'messages':[HumanMessage('Quel est le salaire et le job de Mohamed')]})

In [99]:
load_dotenv(override=True)

True

In [101]:
from langchain_tavily import TavilySearch

In [112]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [113]:
@tool
def searh_web(query: str):
    """
    Search the web for the given query.

    """
    print(f"Searching the web for: {query}")
    result = tavily.invoke({"query": query})
    return result

In [ ]:
agent5 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant"

)

In [118]:
response = agent5.invoke(
    input={'messages':[HumanMessage('What is the weather in Marrakech')]},
    config={'configurable': {'thread_id': '1'}}
)
print(response)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [119]:
from langchain_experimental.tools import PythonREPLTool

In [120]:
repl_tool = PythonREPLTool()

In [121]:
agent6 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="genere et execute du code python en placant le code python et le resultat dans le fichier doc.txt"

)

In [ ]:
res = agent6.invoke(input={"messages":[
    HumanMessage("je veux trier les deux liste [6,8,9,0] et [6,3,7,0] et puis aditionner les deux listes")
]}) 


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************r2YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}